# Copy heads compete with the LM head for backbone capacity — reproduction notebook

Colab無料版（T4）で、README の全結果を再現します。上から順に実行してください。結果はGoogleドライブに保存され、済んだものは飛ばします。

Reproduces all results in the README on a free Colab T4. Run cells top to bottom.

In [ ]:
!nvidia-smi -L
!pip -q install --no-deps tiktoken
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/copy_capacity'
os.makedirs(OUT, exist_ok=True)

## Model / training core

In [ ]:
import math, time, json, os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ================= 設定 =================
def default_cfg():
    return dict(
        vocab_size=50257, d_model=256, n_layers=4, n_heads=4, seq_len=256, d_ff=1024,
        batch_size=32, steps=1500, lr=1e-3, warmup=100, eval_batches=10,
        method="base",      # base / ptr（コピーヘッドのみ） / idcopy（ID照合コピー）
        d_copy=64, code_dim=64, gamma0=2.0,
        seed=0,
    )

# ================= 現代的な小型Transformer =================
# RoPE + RMSNorm(ゲインなし) + QK-norm + ReLU² + 埋め込み非共有 + 出力層ゼロ初期化
def rms(x): return F.rms_norm(x, (x.size(-1),))

def rope(x, T):
    d = x.size(-1); half = d // 2
    inv = 1.0 / (10000 ** (torch.arange(half, device=x.device, dtype=torch.float32) / half))
    ang = torch.arange(T, device=x.device, dtype=torch.float32)[:, None] * inv[None]
    cos, sin = ang.cos().to(x.dtype), ang.sin().to(x.dtype)
    x1, x2 = x[..., :half], x[..., half:]
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1)

class Attn(nn.Module):
    def __init__(s, c):
        super().__init__()
        s.h = c["n_heads"]; D = c["d_model"]
        s.qkv = nn.Linear(D, 3 * D, bias=False); s.o = nn.Linear(D, D, bias=False)
        nn.init.zeros_(s.o.weight)
    def forward(s, x):
        B, T, D = x.shape
        q, k, v = s.qkv(x).split(D, dim=2)
        q, k, v = [t.view(B, T, s.h, D // s.h).transpose(1, 2) for t in (q, k, v)]
        q, k = rope(rms(q), T), rope(rms(k), T)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return s.o(y.transpose(1, 2).reshape(B, T, D))

class MLP(nn.Module):
    def __init__(s, c):
        super().__init__()
        s.a = nn.Linear(c["d_model"], c["d_ff"], bias=False); s.b = nn.Linear(c["d_ff"], c["d_model"], bias=False)
        nn.init.zeros_(s.b.weight)
    def forward(s, x): return s.b(F.relu(s.a(x)).square())

class Block(nn.Module):
    def __init__(s, c):
        super().__init__(); s.a = Attn(c); s.m = MLP(c)
    def forward(s, x):
        x = x + s.a(rms(x)); return x + s.m(rms(x))

class LM(nn.Module):
    def __init__(s, c):
        super().__init__()
        s.c = c; D = c["d_model"]
        s.emb = nn.Embedding(c["vocab_size"], D); nn.init.normal_(s.emb.weight, std=0.02)
        s.blocks = nn.ModuleList([Block(c) for _ in range(c["n_layers"])])
        s.head = nn.Linear(D, c["vocab_size"], bias=False); nn.init.normal_(s.head.weight, std=0.02)
        if c["method"] != "base":
            # コピー用ヘッド：照合用の q,k と「写すかどうか」のゲート
            s.cq = nn.Linear(D, c["d_copy"], bias=False); s.ck = nn.Linear(D, c["d_copy"], bias=False)
            s.gate = nn.Linear(D, 1); nn.init.zeros_(s.gate.weight); nn.init.constant_(s.gate.bias, -2.0)
            g = torch.Generator().manual_seed(1234)
            code = (torch.randint(0, 2, (c["vocab_size"], c["code_dim"]), generator=g).float() * 2 - 1) / math.sqrt(c["code_dim"])
            s.register_buffer("code", code)                      # 固定IDコード（学習しない）
            s.gamma = nn.Parameter(torch.tensor(c["gamma0"] if c["method"] == "idcopy" else 0.0),
                                   requires_grad=(c["method"] == "idcopy"))
    def token_logp(s, idx, tgt):
        """各位置で正解トークンの log 確率を返す (B,T)"""
        h = s.emb(idx)
        for b in s.blocks: h = b(h)
        hn = rms(h)
        logits = s.head(hn).float()
        lp_lm = F.log_softmax(logits, -1).gather(-1, tgt.unsqueeze(-1)).squeeze(-1)
        if s.c["method"] == "base": return lp_lm
        B, T = idx.shape
        q, k = s.cq(hn).float(), s.ck(hn).float()
        sc = q @ k.transpose(1, 2) / math.sqrt(s.c["d_copy"])                 # 文脈による照合
        if s.c["method"] == "idcopy":
            cur = s.code[idx]                                                 # (B,T,m) 今の単語のコード
            prev = torch.zeros_like(cur); prev[:, 1:] = cur[:, :-1]           # 位置jの「直前の単語」のコード
            sc = sc + s.gamma * (cur @ prev.transpose(1, 2))                  # 単語IDの完全一致
        ii = torch.arange(T, device=idx.device)
        mask = (ii[None, :] > ii[:, None])                                    # 未来は見ない（x_0..x_i は既知なので写してよい）
        sc = sc.masked_fill(mask, float("-inf"))
        a = sc.softmax(-1)                                                    # (B,T,T)
        hit = (idx.unsqueeze(1) == tgt.unsqueeze(2)).float()                  # x_j が正解 y_i か
        p_copy = (a * hit).sum(-1)
        g = torch.sigmoid(s.gate(hn).float().squeeze(-1))
        p = (1 - g) * lp_lm.exp() + g * p_copy
        return torch.log(p.clamp_min(1e-12))
    def forward_tokens(s, idx, tgt):
        return -s.token_logp(idx, tgt).mean()
# ================= 学習ループ =================
def get_batch(data, B, L, device):
    ix = torch.randint(len(data) - L - 1, (B,))
    x = torch.stack([data[i:i + L] for i in ix]); y = torch.stack([data[i + 1:i + 1 + L] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def evaluate(model, val, c, device, batches=None):
    model.eval(); out = []
    for _ in range(batches or c["eval_batches"]):
        vx, vy = get_batch(val, c["batch_size"], c["seq_len"], device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            out.append(model.forward_tokens(vx, vy).item())
    return float(np.mean(out))

@torch.no_grad()
def copy_breakdown(model, val, c, device, rank, batches=20):
    """頻度帯 × 既出/初出 ごとの損失"""
    model.eval(); res = {}
    for _ in range(batches):
        x, y = get_batch(val, c["batch_size"], c["seq_len"], device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            nll = (-model.token_logp(x, y)).float()
        # 既出判定：y_i が x_0..x_i の中にあるか
        seen = (x.unsqueeze(1) == y.unsqueeze(2)) & (torch.arange(x.shape[1], device=device)[None, None, :] <= torch.arange(x.shape[1], device=device)[None, :, None])
        seen = seen.any(-1)
        r = rank[y]
        band = torch.where(r < 100, 0, torch.where(r < 1000, 1, 2))
        for bnd in range(3):
            for sn in (True, False):
                m = (band == bnd) & (seen == sn)
                key = f"{['0-100','100-1000','1000+'][bnd]}_{'既出' if sn else '初出'}"
                s_, n_ = res.get(key, (0.0, 0))
                res[key] = (s_ + nll[m].sum().item(), n_ + int(m.sum().item()))
    tot = sum(v[1] for v in res.values())
    return {k: dict(loss=v[0] / max(v[1], 1), frac=v[1] / tot) for k, v in res.items()}

def run(c, train, val, device, rank=None, log=print):
    torch.manual_seed(c["seed"]); random.seed(c["seed"]); np.random.seed(c["seed"])
    model = LM(c).to(device)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=c["lr"], betas=(0.9, 0.95), weight_decay=0.1)
    sched = lambda s: min(1, (s + 1) / c["warmup"]) * 0.5 * (1 + math.cos(math.pi * min(1, s / c["steps"])))
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
    hist = []; t0 = time.time()
    for step in range(c["steps"]):
        for g in opt.param_groups: g["lr"] = c["lr"] * sched(step)
        model.train()
        x, y = get_batch(train, c["batch_size"], c["seq_len"], device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            loss = model.forward_tokens(x, y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        if (step + 1) % 250 == 0 or step + 1 == c["steps"]:
            r = dict(step=step + 1, val_loss=evaluate(model, val, c, device), sec=time.time() - t0)
            if c["method"] == "idcopy": r["gamma"] = float(model.gamma.item())
            hist.append(r)
            log(f"[{c['method']} seed={c['seed']}] step {step+1} val {r['val_loss']:.3f}"
                + (f" γ={r['gamma']:.2f}" if "gamma" in r else "") + f" {r['sec']:.0f}s")
    out = dict(hist=hist)
    if rank is not None: out["breakdown"] = copy_breakdown(model, val, c, device, rank)
    return out


## Data: WikiText-103 (first half of train, GPT-2 tokenizer)

In [ ]:
import tiktoken, numpy as np, torch, os, pandas as pd
from huggingface_hub import hf_hub_download
enc = tiktoken.get_encoding('gpt2')
def tok_parquet(fname):
    p = hf_hub_download('Salesforce/wikitext', fname, repo_type='dataset')
    text = ''.join(pd.read_parquet(p)['text'].tolist())
    arts = text.split('\n = ')                         # 記事ごとに区切る
    out = []
    for i in range(0, len(arts), 20000):
        chunk = []
        for ids in enc.encode_ordinary_batch(arts[i:i+20000], num_threads=8):
            chunk.extend(ids); chunk.append(enc.eot_token)
        out.append(np.array(chunk, dtype=np.uint16))
    return np.concatenate(out)
tr_c = os.path.join(OUT, 'train_tokens.npy'); va_c = os.path.join(OUT, 'val_tokens.npy')
if not os.path.exists(tr_c): np.save(tr_c, tok_parquet('wikitext-103-raw-v1/train-00000-of-00002.parquet'))
if not os.path.exists(va_c): np.save(va_c, tok_parquet('wikitext-103-raw-v1/validation-00000-of-00001.parquet'))
train = torch.from_numpy(np.load(tr_c).astype(np.int64)); val = torch.from_numpy(np.load(va_c).astype(np.int64))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
cnt = torch.bincount(train[:20_000_000], minlength=50257)
order = torch.argsort(-cnt); rank = torch.empty_like(order); rank[order] = torch.arange(len(order)); rank = rank.to(device)
print(f'学習用 {len(train)/1e6:.1f}M / 検証用 {len(val)/1e6:.1f}M トークン / {device}')
print('例:', enc.decode(train[1000:1080].tolist()))

## Methods

In [ ]:
# ===== 質量保存コピー vs 足し算の混合(ptr) vs 対数で足す方式(cpr) =====
import random, json, time
_LM_orig = LM

def heads(m, idx):
    h = m.emb(idx)
    for b in m.blocks: h = b(h)
    hn = rms(h)
    lp = F.log_softmax(m.head(hn).float(), -1)                     # (B,T,V)
    T = idx.shape[1]
    sc = m.cq(hn).float() @ m.ck(hn).float().transpose(1, 2) / math.sqrt(m.c['d_copy'])
    ii = torch.arange(T, device=idx.device)
    sc = sc.masked_fill(ii[None, :] > ii[:, None], float('-inf'))
    g = torch.sigmoid(m.gate(hn).float().squeeze(-1))
    return lp, sc, g

def logp(m, idx, tgt, kind):
    lp, sc, g = heads(m, idx)
    lp_y = lp.gather(-1, tgt.unsqueeze(-1)).squeeze(-1)
    if kind == 'base': return lp_y
    T = idx.shape[1]
    ii = torch.arange(T, device=idx.device)
    causal = ii[None, :] <= ii[:, None]                             # (T,T)  j<=i
    hit = (idx.unsqueeze(1) == tgt.unsqueeze(2)) & causal           # x_j == y_i
    if kind == 'cpr':
        # 文中の単語 v に、得点 log(1 + sum_j exp(sc_ij)[x_j=v]) を足して1つのsoftmaxで正規化（対数空間で足す既存方式）
        # 語彙サイズのテンソルを作らずに計算：正規化定数 = log(1 + sum_j exp(sc_ij) p_LM(x_j))
        e = sc.exp()                                                # (B,T,T)、未来は0
        lp_x = lp.gather(-1, idx.unsqueeze(1).expand(-1, T, -1))    # (B,T,T)
        b_y = (e * hit.float()).sum(-1)                             # 正解への追加分
        Z = torch.log1p((e * lp_x.exp()).sum(-1))
        return lp_y + torch.log1p(b_y) - Z
    a = sc.softmax(-1)
    p_copy_y = (a * hit.float()).sum(-1)
    p_lm_y = lp_y.exp()
    if kind == 'ptr':
        return torch.log(((1 - g) * p_lm_y + g * p_copy_y).clamp_min(1e-12))
    # masscopy：P_C = sum_{v in C} p_LM(v)。重複は「位置iまでの出現回数」で割って数える
    lp_x = lp.gather(-1, idx.unsqueeze(1).expand(-1, T, -1))        # (B,T,T): p_LM(x_j) at position i
    same = (idx.unsqueeze(2) == idx.unsqueeze(1)).float()           # (B,T,T): x_j == x_k
    cnt = torch.einsum('bjk,ik->bij', same, causal.float())         # (B,T,T): 位置iまでの x_j の出現回数
    w = causal.float()[None] / cnt.clamp(min=1)
    P_C = (lp_x.exp() * w).sum(-1)                                  # (B,T)
    in_C = hit.any(-1)
    p = torch.where(in_C, (1 - g) * p_lm_y + g * P_C * p_copy_y, p_lm_y)
    return torch.log(p.clamp_min(1e-12))

@torch.no_grad()
def evaluate_kind(m, c, kind, batches=20):
    m.eval(); tot = []; new = []
    for _ in range(batches):
        x, y = get_batch(val, c['batch_size'], c['seq_len'], device)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            nll = -logp(m, x, y, 'ptr' if kind == 'ptr_aux' else kind).float()
        T = x.shape[1]; ii = torch.arange(T, device=device)
        seen = ((x.unsqueeze(1) == y.unsqueeze(2)) & (ii[None, None, :] <= ii[None, :, None])).any(-1)
        tot.append(nll.mean().item()); new.append(nll[~seen].mean().item())
    return float(np.mean(tot)), float(np.mean(new))

def run_kind(kind, seed=0, steps=1500):
    path = os.path.join(OUT, f'mass_{kind}_s{seed}.json')
    if os.path.exists(path):
        print('済み:', kind); d = json.load(open(path)); return d['val'], d['new']
    c = default_cfg(); c.update(method='ptr', seed=seed, steps=steps)
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    m = _LM_orig(c).to(device)
    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=c['lr'], betas=(0.9, 0.95), weight_decay=0.1)
    sched = lambda s_: min(1, (s_ + 1) / c['warmup']) * 0.5 * (1 + math.cos(math.pi * min(1, s_ / steps)))
    scaler = torch.amp.GradScaler('cuda'); t0 = time.time()
    for step in range(steps):
        for gp in opt.param_groups: gp['lr'] = c['lr'] * sched(step)
        m.train(); x, y = get_batch(train, c['batch_size'], c['seq_len'], device)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            if kind == 'ptr_aux':
                lp_, sc_, g_ = heads(m, x)
                lp_y_ = lp_.gather(-1, y.unsqueeze(-1)).squeeze(-1)
                T_ = x.shape[1]; ii_ = torch.arange(T_, device=device)
                hit_ = (x.unsqueeze(1) == y.unsqueeze(2)) & (ii_[None, :] <= ii_[:, None])
                pc_ = (sc_.softmax(-1) * hit_.float()).sum(-1)
                lp_mix_ = torch.log(((1 - g_) * lp_y_.exp() + g_ * pc_).clamp_min(1e-12))
                loss = -lp_mix_.mean() + AUX_LAMBDA * (-lp_y_.mean())   # 普通の出力層だけの損失も足す
            else:
                loss = -logp(m, x, y, kind).mean()
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); scaler.step(opt); scaler.update()
        if (step + 1) % 250 == 0:
            print(f'[{kind}] step {step+1} loss {loss.item():.3f} {time.time()-t0:.0f}s')
    v, n = evaluate_kind(m, c, kind)
    json.dump(dict(val=v, new=n), open(path, 'w'))
    return v, n

AUX_LAMBDA = 1.0


In [ ]:

def eval_fwd(fwd, c):
    tot, new = [], []
    with torch.no_grad():
        for _ in range(20):
            x, y = get_batch(val, c['batch_size'], c['seq_len'], device)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                lp, _, hit = fwd(x, y)
            nll = -lp.float(); seen = hit.any(-1)
            tot.append(nll.mean().item()); new.append(nll[~seen].mean().item())
    return float(np.mean(tot)), float(np.mean(new))

def run_variant(variant, seed, steps=1500, tap=2, lam=1.0):
    """variant: 'iso'（照合とゲートの勾配を止める＋aux） / 'mid'（照合を途中層から＋aux：提案法）"""
    path = os.path.join(OUT, f'{variant}_s{seed}.json')
    if os.path.exists(path):
        print('済み:', variant, seed); d = json.load(open(path)); return d['val'], d['new']
    c = default_cfg(); c.update(method='ptr', seed=seed, steps=steps)
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    m = _LM_orig(c).to(device)
    def fwd(x, y):
        h = m.emb(x); hm = None
        for l, b in enumerate(m.blocks):
            h = b(h)
            if l == tap - 1: hm = rms(h)
        hn = rms(h)
        lp_y = F.log_softmax(m.head(hn).float(), -1).gather(-1, y.unsqueeze(-1)).squeeze(-1)
        src_q = hn.detach() if variant == 'iso' else hm
        src_g = hn.detach() if variant == 'iso' else hn
        T = x.shape[1]; ii = torch.arange(T, device=device)
        sc = m.cq(src_q).float() @ m.ck(src_q).float().transpose(1, 2) / math.sqrt(c['d_copy'])
        sc = sc.masked_fill(ii[None, :] > ii[:, None], float('-inf'))
        hit = (x.unsqueeze(1) == y.unsqueeze(2)) & (ii[None, :] <= ii[:, None])
        pc = (sc.softmax(-1) * hit.float()).sum(-1)
        g = torch.sigmoid(m.gate(src_g).float().squeeze(-1))
        return torch.log(((1 - g) * lp_y.exp() + g * pc).clamp_min(1e-12)), lp_y, hit
    params = [p for p in m.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=c['lr'], betas=(0.9, 0.95), weight_decay=0.1)
    sched = lambda s_: min(1, (s_ + 1) / c['warmup']) * 0.5 * (1 + math.cos(math.pi * min(1, s_ / steps)))
    scaler = torch.amp.GradScaler('cuda'); t0 = time.time()
    for step in range(steps):
        for gp in opt.param_groups: gp['lr'] = c['lr'] * sched(step)
        m.train(); x, y = get_batch(train, c['batch_size'], c['seq_len'], device)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            lp, lp_y, _ = fwd(x, y)
            loss = -lp.mean() + lam * (-lp_y.mean())
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 1.0); scaler.step(opt); scaler.update()
        if (step + 1) % 250 == 0: print(f'[{variant} s{seed}] step {step+1} loss {loss.item():.3f} {time.time()-t0:.0f}s')
    m.eval(); v, n = eval_fwd(fwd, c)
    json.dump(dict(val=v, new=n), open(path, 'w')); return v, n


In [ ]:
# ===== 論文と同じ設計（〈copy〉ゲート＋sink）を比較相手にする（約20分） =====
# cplm      : Copy-Paste LM と同じ設計（最終層から照合、混合の損失のみ）
# cplm_ours : 同じ設計に、私たちの2つの工夫（途中層からの照合＋普通の出力層の損失）を足したもの
def run_cplm(seed, ours, steps=1500, tap=2, lam=1.0):
    name = f"cplm{'_ours' if ours else ''}_s{seed}"
    path = os.path.join(OUT, name + '.json')
    if os.path.exists(path):
        print('済み:', name); d = json.load(open(path)); return d['val'], d['new']
    c = default_cfg(); c.update(method='ptr', seed=seed, steps=steps)
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    m = _LM_orig(c).to(device)
    D = c['d_model']
    m.copy_logit = nn.Linear(D, 1, bias=True).to(device)       # 出力層に足す〈copy〉の得点
    nn.init.zeros_(m.copy_logit.weight); nn.init.zeros_(m.copy_logit.bias)
    m.sink = nn.Parameter(torch.zeros(c['d_copy'], device=device))   # 照合に足す「写さない」用のキー
    def fwd(x, y):
        h = m.emb(x); hm = None
        for l, b in enumerate(m.blocks):
            h = b(h)
            if l == tap - 1: hm = rms(h)
        hn = rms(h)
        logits = m.head(hn).float()                                # (B,T,V)
        cl = m.copy_logit(hn).float()                               # (B,T,1)
        lse_all = torch.logsumexp(torch.cat([logits, cl], -1), -1)  # V+1 個での正規化
        c_t = torch.exp(cl.squeeze(-1) - lse_all)                   # 〈copy〉の確率
        lp_lm = F.log_softmax(logits, -1).gather(-1, y.unsqueeze(-1)).squeeze(-1)   # p_LM は〈copy〉を除いて正規化
        src = hm if ours else hn
        T = x.shape[1]; ii = torch.arange(T, device=device)
        q = m.cq(src).float(); k = m.ck(src).float()
        sc = q @ k.transpose(1, 2) / math.sqrt(c['d_copy'])
        sc = sc.masked_fill(ii[None, :] > ii[:, None], float('-inf'))
        s_sc = (q @ m.sink.float()) / math.sqrt(c['d_copy'])        # (B,T) sink の得点
        a = torch.cat([sc, s_sc.unsqueeze(-1)], -1).softmax(-1)
        hit = (x.unsqueeze(1) == y.unsqueeze(2)) & (ii[None, :] <= ii[:, None])
        copy_mass = (a[..., :T] * hit.float()).sum(-1)              # (1-s) * p_CP(y)
        p = (1 - c_t * (1 - a[..., T])) * lp_lm.exp() + c_t * copy_mass   # λ = c(1-s)
        return torch.log(p.clamp_min(1e-12)), lp_lm, hit
    params = [p for p in m.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=c['lr'], betas=(0.9, 0.95), weight_decay=0.1)
    sched = lambda s_: min(1, (s_ + 1) / c['warmup']) * 0.5 * (1 + math.cos(math.pi * min(1, s_ / steps)))
    scaler = torch.amp.GradScaler('cuda'); t0 = time.time()
    for step in range(steps):
        for gp in opt.param_groups: gp['lr'] = c['lr'] * sched(step)
        m.train(); x, y = get_batch(train, c['batch_size'], c['seq_len'], device)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            lp, lp_lm, _ = fwd(x, y)
            loss = -lp.mean() + (lam * (-lp_lm.mean()) if ours else 0.0)
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 1.0); scaler.step(opt); scaler.update()
        if (step + 1) % 250 == 0: print(f'[{name}] step {step+1} loss {loss.item():.3f} {time.time()-t0:.0f}s')
    m.eval(); tot, new = [], []
    with torch.no_grad():
        for _ in range(20):
            x, y = get_batch(val, c['batch_size'], c['seq_len'], device)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                lp, _, hit = fwd(x, y)
            nll = -lp.float(); seen = hit.any(-1)
            tot.append(nll.mean().item()); new.append(nll[~seen].mean().item())
    v, n = float(np.mean(tot)), float(np.mean(new))
    json.dump(dict(val=v, new=n), open(path, 'w')); return v, n



## Run all

In [ ]:
# ===== 全条件を実行して表にする（1本あたり約5分、cplm系は約8分） =====
SEEDS = [0, 1]          # 時間があれば [0, 1, 2] に
ALL = {}
for s in SEEDS:
    ALL[('base', s)] = run_kind('base', seed=s)
    ALL[('ptr', s)] = run_kind('ptr', seed=s)
    ALL[('ptr_aux', s)] = run_kind('ptr_aux', seed=s)
    ALL[('iso', s)] = run_variant('iso', s)
    ALL[('mid', s)] = run_variant('mid', s)
    ALL[('cplm', s)] = run_cplm(s, False)
    ALL[('cplm_ours', s)] = run_cplm(s, True)
names = ['base', 'ptr', 'ptr_aux', 'iso', 'mid', 'cplm', 'cplm_ours']
print('\n条件        ' + '  '.join(f'seed{s}' for s in SEEDS) + '   平均     初出の損失')
for n in names:
    vs = [ALL[(n, s)][0] for s in SEEDS]; ns = [ALL[(n, s)][1] for s in SEEDS]
    print(f'{n:10s}  ' + '  '.join(f'{v:.4f}' for v in vs) + f'   {np.mean(vs):.4f}   {np.mean(ns):.3f}')
